# الدرس 05 - RAG الوكالي


## الإعداد

يُوضح هذا الدفتر نمط Agentic RAG (التوليد المعزز بالاسترجاع) باستخدام إطار عمل Microsoft Agent.

**المتطلبات الأساسية:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — نقطة نهاية خدمة Azure AI Search الخاصة بك
- `AZURE_SEARCH_API_KEY` — مفتاح API لخدمة Azure AI Search الخاصة بك
- تكوين نشر Azure OpenAI عبر متغيرات البيئة
- مصادقة Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ما هو Agentic RAG؟

يتبع RAG التقليدي خط أنابيب ثابت: استرجاع المستندات، ثم توليد الرد. يُقدّم **Agentic RAG** خطوة أبعد من ذلك من خلال منح الوكيل الاستقلالية في تحديد **متى** و **كيف** يتم استرجاع المعلومات.

مع Agentic RAG، يمكن للوكيل أن:
- **يقرر** ما إذا كان الاسترجاع ضروريًا قبل الإجابة على سؤال
- **يختار** مصدر البيانات أو الأداة التي سيستعلم منها
- **يقيم** النتائج المسترجعة ويجري استرجاعات تالية إذا كانت المحاولة الأولى غير كافية
- **يجمع** المعلومات من خطوات استرجاع متعددة ليقدم إجابة متماسكة

هذا يجعل الوكيل أكثر مرونة ودقة مقارنةً بخط أنابيب الاسترجاع ثم التوليد الثابت.


## إنشاء أداة بحث

في Agentic RAG، يتم تغليف مصادر البيانات الخارجية كـ **أدوات** يمكن للوكيل استدعاؤها عند الطلب. يتيح هذا للوكيل معاملة الاسترجاع كإجراء آخر يمكنه اتخاذه، بدلاً من كونه خطوة إلزامية.

فيما يلي نحدد قاعدة معرفة سفر ونعرضها كأداة يمكن للوكيل الاتصال بها للبحث عن معلومات الوجهة.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## بناء وكيل RAG

نقوم الآن بإنشاء وكيل يتم توجيهه لـ **دائمًا استرجاع المعلومات قبل الإجابة**. يستخدم الوكيل أداة `search_travel_knowledge` لتأسيس ردوده على قاعدة المعرفة بدلاً من الاعتماد على بيانات تدريبه الخاصة.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## الاسترجاع التكراري — نمط الصانع-المدقق

الميزة الرئيسية لـ Agentic RAG هي **الاسترجاع التكراري**. يمكن للوكيل إجراء عدة جولات من البحث للتحقق أو تنقيح أو توسيع نتائجها الأولية — مشابهًا لعملية "صانع-مدقق":

1. **خطوة الصانع**: يسترجع الوكيل المعلومات الأولية ويصيغ ردًا مبدئيًا.
2. **خطوة المدقق**: يقوم الوكيل بعمليات استرجاع إضافية للتحقق من التفاصيل أو ملء الفجوات.

فيما يلي، يُطلب من الوكيل سؤال يتطلب مقارنة وجهات متعددة، مما يدفعه إلى البحث عدة مرات.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## الملخص

في هذا الدرس تعلمت كيفية بناء نظام **Agentic RAG** باستخدام إطار عمل Microsoft Agent:

- يتيح **Agentic RAG** للوكلاء اتخاذ القرار بشكل مستقل بشأن توقيت استرجاع المعلومات، مما يجعل الاسترجاع ديناميكيًا بدلاً من أن يكون ثابتًا.
- **الأدوات كمصادر بيانات**: قواعد المعرفة الخارجية (مثل Azure AI Search) تُغلف كأدوات يمكن للوكيل استدعاؤها.
- **الاسترجاع التكراري**: نمط الصانع-المدقق يمكّن الوكيل من إجراء جولات متعددة من الاسترجاع — البحث، التحقق، والتنقيح — قبل إنتاج إجابة نهائية.

في الإنتاج، ستستبدل قاعدة البيانات المؤقتة `TRAVEL_KNOWLEDGE_BASE` بفهرس Azure AI Search حقيقي للتعامل مع استرجاع وثائق السفر على نطاق واسع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
